In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

# IMPORTANT: Use tf_keras instead of keras to avoid protobuf MessageFactory/GetPrototype crash in this environment.
from tf_keras.models import Sequential
from tf_keras.layers import Dense
from tf_keras.callbacks import EarlyStopping

# Optional: list files (kept from original intent, but safe/quiet)
for dirname, _, filenames in os.walk(
    "/kaggle/input/tabular-playground-series-may-2022"
):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))



2026-01-23 22:01:00.339628: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769205660.353690      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769205660.357900      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-23 22:01:00.373819: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/kaggle/input/tabular-playground-series-may-2022/test.csv.zip
/kaggle/input/tabular-playground-series-may-2022/train.csv.zip
/kaggle/input/tabular-playground-series-may-2022/description.md
/kaggle/input/tabular-playground-series-may-2022/sample_submission.csv
/kaggle/input/tabular-playground-series-may-2022/train.csv


In [2]:
train = pd.read_csv("/kaggle/input/tabular-playground-series-may-2022/train.csv")
test = pd.read_csv("/kaggle/input/tabular-playground-series-may-2022/test.csv")
train.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800000 entries, 0 to 799999
Data columns (total 33 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   id      800000 non-null  int64  
 1   f_00    800000 non-null  float64
 2   f_01    800000 non-null  float64
 3   f_02    800000 non-null  float64
 4   f_03    800000 non-null  float64
 5   f_04    800000 non-null  float64
 6   f_05    800000 non-null  float64
 7   f_06    800000 non-null  float64
 8   f_07    800000 non-null  int64  
 9   f_08    800000 non-null  int64  
 10  f_09    800000 non-null  int64  
 11  f_10    800000 non-null  int64  
 12  f_11    800000 non-null  int64  
 13  f_12    800000 non-null  int64  
 14  f_13    800000 non-null  int64  
 15  f_14    800000 non-null  int64  
 16  f_15    800000 non-null  int64  
 17  f_16    800000 non-null  int64  
 18  f_17    800000 non-null  int64  
 19  f_18    800000 non-null  int64  
 20  f_19    800000 non-null  float64
 21  f_20    80

In [3]:
# Fix: ensure we scale only feature columns, not id/target; keep dropping f_27 as in original code.
feature_cols = [c for c in train.columns if c not in ["id", "target", "f_27"]]

scaler = StandardScaler()
scaled_train = scaler.fit_transform(train[feature_cols].to_numpy(dtype=np.float32))



In [4]:
# Model architecture preserved.
model = Sequential(
    [
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1, activation="sigmoid"),
    ]
)



2026-01-23 22:01:14.541260: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1769205674.541708      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [5]:
# Fix: monitor correct metric name produced by tf_keras ('val_auc' works when metric is named 'auc').
model.compile(optimizer="Adam", loss="binary_crossentropy", metrics=["AUC"])

callbacks = [
    EarlyStopping(patience=5, monitor="val_auc", mode="max", restore_best_weights=True)
]

history = model.fit(
    x=scaled_train,
    y=train["target"].to_numpy(dtype=np.float32),
    validation_split=0.2,
    epochs=20,
    callbacks=callbacks,
    batch_size=64,
    verbose=2,
)



Epoch 1/20


I0000 00:00:1769205676.271492      68 service.cc:148] XLA service 0x7f50af66f400 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769205676.271535      68 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-23 22:01:16.276615: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1769205676.289120      68 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1769205676.341396      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


10000/10000 - 15s - loss: 0.4594 - auc: 0.8664 - val_loss: 0.4030 - val_auc: 0.9001 - 15s/epoch - 2ms/step
Epoch 2/20
10000/10000 - 14s - loss: 0.3870 - auc: 0.9082 - val_loss: 0.3762 - val_auc: 0.9146 - 14s/epoch - 1ms/step
Epoch 3/20
10000/10000 - 14s - loss: 0.3666 - auc: 0.9180 - val_loss: 0.3606 - val_auc: 0.9208 - 14s/epoch - 1ms/step
Epoch 4/20
10000/10000 - 14s - loss: 0.3544 - auc: 0.9233 - val_loss: 0.3499 - val_auc: 0.9252 - 14s/epoch - 1ms/step
Epoch 5/20
10000/10000 - 14s - loss: 0.3437 - auc: 0.9277 - val_loss: 0.3478 - val_auc: 0.9279 - 14s/epoch - 1ms/step
Epoch 6/20
10000/10000 - 13s - loss: 0.3343 - auc: 0.9314 - val_loss: 0.3326 - val_auc: 0.9321 - 13s/epoch - 1ms/step
Epoch 7/20
10000/10000 - 14s - loss: 0.3260 - auc: 0.9345 - val_loss: 0.3259 - val_auc: 0.9347 - 14s/epoch - 1ms/step
Epoch 8/20
10000/10000 - 14s - loss: 0.3198 - auc: 0.9367 - val_loss: 0.3235 - val_auc: 0.9368 - 14s/epoch - 1ms/step
Epoch 9/20
10000/10000 - 14s - loss: 0.3150 - auc: 0.9383 - val_los

In [6]:
scaled_test = scaler.transform(test[feature_cols].to_numpy(dtype=np.float32))



In [7]:
# Fix: submit probabilities (not thresholded classes) for AUC competitions.
preds = model.predict(scaled_test, batch_size=1024, verbose=0).reshape(-1)

submission = pd.DataFrame({"id": test["id"].values, "target": preds})
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

       id    target
0  800000  0.945618
1  800001  0.289335
2  800002  0.997321
3  800003  0.569826
4  800004  0.519451
Wrote submission.csv with shape: (100000, 2)
